# Playground — DeCo-VTON try-on (AutoMasker)

Person + cloth only (no dataset agnostic). Builds the agnostic mask with AutoMasker (DensePose + SCHP), then runs [DeCo-VTON](https://huggingface.co/levinna/DeCo-VTON).

**Prerequisites** (CUDA recommended; Detectron2 DensePose is not reliable on MPS):

```bash
.venv/bin/python -m pip install opencv-python
.venv/bin/python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'
.venv/bin/python -m pip install 'git+https://github.com/facebookresearch/detectron2.git#subdirectory=projects/DensePose'
```

CatVTON preprocess weights (`DensePose/`, `SCHP/`) are downloaded from `zhengchong/CatVTON` on first `load_automasker` call.

In [ ]:
import torch

from fashion_diffusion.config import CATVTON_HF_REPO, DECOVTON_HF_REPO, DECOVTON_HF_SUBFOLDER, DEVICE
from fashion_diffusion.paths import DATA_DIR
from fashion_diffusion.agnostic_mask import MASK_DEVICE, generate_agnostic_mask, load_automasker
from fashion_diffusion.decovton import (
    load_pipeline,
    prepare_inputs_from_mask,
    run_tryon,
    show_condition_and_mask,
    show_person_mask,
    show_result,
)

In [ ]:
person_filename = "10336_00.jpg"
# cloth_filename = "10740_00.jpg"
cloth_filename = "10336_00.jpg"
split = "train"

person_path = DATA_DIR / split / "image" / person_filename
cloth_path = DATA_DIR / split / "cloth" / cloth_filename

mask_type = "upper"  # one of: upper, lower, overall, inner, outer
dtype = torch.float32
num_inference_steps = 5

print("==== Input paths:")
print(f"\tperson_path: {person_path}")
print(f"\tcloth_path: {cloth_path}")
print(f"\tDECOVTON_HF_REPO: {DECOVTON_HF_REPO}")
print(f"\tDECOVTON_HF_SUBFOLDER: {DECOVTON_HF_SUBFOLDER}")
print(f"\tCATVTON_HF_REPO: {CATVTON_HF_REPO}")
print(f"\tmask_type: {mask_type}")
print(f"\tMASK_DEVICE: {MASK_DEVICE}")
print(f"\tDEVICE (try-on): {DEVICE}")
print(f"\tnum_inference_steps: {num_inference_steps}")

In [ ]:
automasker = load_automasker(CATVTON_HF_REPO)
mask_pil = generate_agnostic_mask(automasker, person_path, mask_type=mask_type)
show_person_mask(person_path, mask_pil)

In [ ]:
condition, mask = prepare_inputs_from_mask(person_path, cloth_path, mask_pil)
show_condition_and_mask(condition, mask)

In [ ]:
pipe = load_pipeline(DECOVTON_HF_REPO, DECOVTON_HF_SUBFOLDER, dtype)
result = run_tryon(pipe, condition, mask, num_inference_steps=num_inference_steps)
show_result(person_path, cloth_path, result)

In [ ]:
from fashion_diffusion.metrics import print_pair_metrics

print_pair_metrics(person_path, cloth_path, result)